# 02 — Validate Bronze

## Objective

Validate that the Bronze tables were correctly created and contain the expected structure and technical metadata.

## This notebook performs

- Selection of the project schema.
- Definition of expected Bronze tables.
- Definition of expected business columns for each table.
- Validation of table existence.
- Validation of row counts.
- Validation of expected business columns.
- Validation of Bronze metadata columns:
  - `source_file`
  - `ingestion_timestamp`
  - `bronze_load_id`
- Creation of a persistent Bronze quality log table.

## Notes

This notebook performs structural validation only. It does not clean or reject business-level data quality issues such as invalid emails, negative hours, duplicated employees or inconsistent regions. Those rules are handled in the Silver transformation layer.

In [0]:
# Import Spark functions and Python utilities used for validation logging.

from datetime import datetime
from pyspark.sql import functions as F

# Use the project schema where the Bronze tables were created.

spark.sql("USE SCHEMA workbridge")

DataFrame[]

In [0]:
# Define the expected Bronze tables and their required business columns.
# These are the columns that should exist after the raw files were ingested.

expected_tables = {
    "bronze_headcount": [
        "employee_id",
        "employee_name",
        "employee_email",
        "gender",
        "country",
        "region",
        "department",
        "role",
        "seniority",
        "hire_date",
        "termination_date",
        "employment_status",
    ],
    "bronze_assignments": [
        "assignment_id",
        "employee_id",
        "client_id",
        "project_id",
        "assignment_start_date",
        "assignment_end_date",
        "allocation_percentage",
        "assignment_status",
    ],
    "bronze_hours": [
        "period",
        "employee_id",
        "client_id",
        "project_id",
        "available_hours",
        "worked_hours",
        "billable_hours",
        "non_billable_hours",
        "overtime_hours",
    ],
    "bronze_costs": [
        "period",
        "employee_id",
        "region",
        "salary_cost",
        "benefits_cost",
        "total_cost",
        "currency",
    ],
    "bronze_goals": [
        "period",
        "client_id",
        "region",
        "target_turnover_rate",
        "target_utilization_rate",
        "target_cost",
        "target_billable_hours",
    ],
    "bronze_clients": [
        "client_id",
        "client_name",
        "industry",
        "client_region",
        "account_manager",
        "contract_type",
    ],
}

# These metadata columns should exist in every Bronze table.
# They were added in the 01_ingest_bronze notebook.

metadata_columns = [
    "source_file",
    "ingestion_timestamp",
    "bronze_load_id",
]

In [0]:
# This list will store the result of each validation check.
# At the end of the notebook, it will be converted into a Spark DataFrame
# and saved as a Delta table.

validation_results = []

In [0]:
def add_validation_result(table_name, check_name, status, records_count=None, message=""):
    """
    Add a validation result to the validation_results list.

    Parameters:
    - table_name: name of the table being validated.
    - check_name: name of the validation check.
    - status: PASS or FAIL.
    - records_count: number of records involved in the check.
    - message: short explanation of the result.
    """

    validation_results.append({
        "validation_timestamp": datetime.now(),
        "table_name": table_name,
        "check_name": check_name,
        "status": status,
        "records_count": records_count,
        "message": message,
    })

In [0]:
# Validate each expected Bronze table.
# The checks performed here are structural checks:
# - table exists;
# - table has rows;
# - expected business columns exist;
# - expected metadata columns exist.

for table_name, expected_columns in expected_tables.items():

    try:
        # Try to read the table from the current schema.
        df = spark.table(table_name)

        # If the table can be read, it exists.
        add_validation_result(
            table_name=table_name,
            check_name="table_exists_check",
            status="PASS",
            message="Table exists."
        )

        # Check if the table contains records.
        row_count = df.count()

        if row_count > 0:
            add_validation_result(
                table_name=table_name,
                check_name="row_count_check",
                status="PASS",
                records_count=row_count,
                message="Table contains records."
            )
        else:
            add_validation_result(
                table_name=table_name,
                check_name="row_count_check",
                status="FAIL",
                records_count=row_count,
                message="Table is empty."
            )

        # Check if all expected business columns exist.
        actual_columns = df.columns
        missing_business_columns = [
            column for column in expected_columns
            if column not in actual_columns
        ]

        if len(missing_business_columns) == 0:
            add_validation_result(
                table_name=table_name,
                check_name="expected_columns_check",
                status="PASS",
                records_count=row_count,
                message="All expected business columns exist."
            )
        else:
            add_validation_result(
                table_name=table_name,
                check_name="expected_columns_check",
                status="FAIL",
                records_count=row_count,
                message=f"Missing business columns: {missing_business_columns}"
            )

        # Check if all Bronze metadata columns exist.
        missing_metadata_columns = [
            column for column in metadata_columns
            if column not in actual_columns
        ]

        if len(missing_metadata_columns) == 0:
            add_validation_result(
                table_name=table_name,
                check_name="metadata_columns_check",
                status="PASS",
                records_count=row_count,
                message="All Bronze metadata columns exist."
            )
        else:
            add_validation_result(
                table_name=table_name,
                check_name="metadata_columns_check",
                status="FAIL",
                records_count=row_count,
                message=f"Missing metadata columns: {missing_metadata_columns}"
            )

    except Exception as error:
        # If the table cannot be read, it probably does not exist
        # or there is an access/storage issue.

        add_validation_result(
            table_name=table_name,
            check_name="table_exists_check",
            status="FAIL",
            records_count=None,
            message=f"Table could not be read: {str(error)}"
        )

In [0]:
# Convert validation results into a Spark DataFrame.

bronze_quality_log_df = spark.createDataFrame(validation_results)

display(bronze_quality_log_df)

check_name,message,records_count,status,table_name,validation_timestamp
table_exists_check,Table exists.,null,PASS,bronze_headcount,2026-05-09T12:45:01.325Z
row_count_check,Table contains records.,508,PASS,bronze_headcount,2026-05-09T12:45:01.844Z
expected_columns_check,All expected business columns exist.,508,PASS,bronze_headcount,2026-05-09T12:45:02.100Z
metadata_columns_check,All Bronze metadata columns exist.,508,PASS,bronze_headcount,2026-05-09T12:45:02.100Z
table_exists_check,Table exists.,null,PASS,bronze_assignments,2026-05-09T12:45:02.100Z
row_count_check,Table contains records.,555,PASS,bronze_assignments,2026-05-09T12:45:02.663Z
expected_columns_check,All expected business columns exist.,555,PASS,bronze_assignments,2026-05-09T12:45:02.851Z
metadata_columns_check,All Bronze metadata columns exist.,555,PASS,bronze_assignments,2026-05-09T12:45:02.851Z
table_exists_check,Table exists.,null,PASS,bronze_hours,2026-05-09T12:45:02.852Z
row_count_check,Table contains records.,4154,PASS,bronze_hours,2026-05-09T12:45:03.230Z


In [0]:
# Save validation results as a Delta table.
# This creates a persistent quality log that can be reviewed later.

bronze_quality_log_df.write.mode("overwrite").format("delta").saveAsTable("bronze_quality_log")

In [0]:
# Show validation summary by status.

summary_df = (
    bronze_quality_log_df
    .groupBy("status")
    .agg(F.count("*").alias("checks_count"))
)

display(summary_df)

status,checks_count
PASS,24
